In [1]:
import pandas as pd
df = pd.read_csv('train.csv')

In [2]:
from IPython.display import display
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    display(df[col].value_counts(dropna=False))

health_condition
at-risk      592561
unhealthy     57724
fit           39803
Name: count, dtype: int64

diet_type
veg         231432
balanced    226888
non-veg     224867
NaN           6901
Name: count, dtype: int64

stress_level
medium    261819
high      177750
low       167708
NaN        82811
Name: count, dtype: int64

sleep_quality
average    213948
poor       212166
good       205643
NaN         58331
Name: count, dtype: int64

physical_activity_level
moderate     221041
sedentary    219784
active       212642
NaN           36621
Name: count, dtype: int64

smoking_alcohol
yes           223730
no            219791
occasional    217985
NaN            28582
Name: count, dtype: int64

gender
male      237756
female    224016
other     206943
NaN        21373
Name: count, dtype: int64

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector

#　對ｙ進行labelencode

# num_transformer 
num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# cat_transformer
cat_trasformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# combine transformer
preprocessor =  ColumnTransformer(
    transformers = [
        ('num', num_transformer, make_column_selector(dtype_include=['number'])),
        ('cat', cat_trasformer, make_column_selector(dtype_include=['object']))
    ]
)

preprocessor.set_output(transform='pandas')

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [4]:
# define the data
from sklearn.preprocessing import LabelEncoder
import gc
# X
X = df.loc[:, [c for c in df.columns if c not in ['id', 'health_condition']]]
# y
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['health_condition'])
# split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [5]:
print(pd.Series(y).value_counts())

0    592561
2     57724
1     39803
Name: count, dtype: int64


In [4]:
# build the XGBoost model
from xgboost import XGBClassifier
mod_baseline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('XGBclassifier', XGBClassifier( 
        n_estimators = 100,
        learning_rate = 0.1,
        max_depth = 6, 
        subsample = 1.0,
        colsample_bytree = 1.0,
        gamma = 0, 
        reg_alpha = 0,
        reg_lambda = 1,
        
        tree_method = 'hist'
        
    ))
])

mod_baseline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('XGBclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [5]:
#　report
from sklearn.metrics import classification_report
train_predict = mod_baseline.predict(X_train)
test_predict = mod_baseline.predict(X_test)
report_train = classification_report(y_train, train_predict)
report_test = classification_report(y_test, test_predict)
print(f'train: {report_train}\n test: {report_test}')

train:               precision    recall  f1-score   support

           0       0.97      0.99      0.98    473940
           1       0.94      0.83      0.88     31844
           2       0.96      0.80      0.88     46286

    accuracy                           0.97    552070
   macro avg       0.96      0.88      0.91    552070
weighted avg       0.97      0.97      0.97    552070

 test:               precision    recall  f1-score   support

           0       0.97      0.99      0.98    118621
           1       0.94      0.82      0.88      7959
           2       0.95      0.80      0.87     11438

    accuracy                           0.97    138018
   macro avg       0.95      0.87      0.91    138018
weighted avg       0.97      0.97      0.97    138018



#### 模型結果解釋
整體來看，模型較無過擬合風險，因為訓練和測試的準確度近乎相等，代表模型有一定預測未知資料的能力。然而觀察recall指標，預測1、2的招回率明顯偏低，模型在預測1、2較為保守，容易將1、2樣本誤判為0。觀察樣本分布，主要原因可能是樣本分布不均衡導致。
後續優化目標調整樣本權重、調整樹結構與學習率

In [6]:
# 調整樣本權重
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
mod2 = clone(mod_baseline)
mod2 = mod2.fit(
    X_train, y_train,
    XGBclassifier__sample_weight=compute_sample_weight('balanced', y_train)
)

In [7]:
mod2_train_predict = mod2.predict(X_train)
mod2_test_predict = mod2.predict(X_test)
report_mod2_train = classification_report(y_train, mod2_train_predict)
report_mod2_test = classification_report(y_test, mod2_test_predict)
print(f'train: {report_mod2_train}\n test: {report_mod2_test}')

train:               precision    recall  f1-score   support

           0       0.99      0.94      0.96    473940
           1       0.73      0.95      0.83     31844
           2       0.69      0.97      0.81     46286

    accuracy                           0.94    552070
   macro avg       0.81      0.95      0.87    552070
weighted avg       0.95      0.94      0.94    552070

 test:               precision    recall  f1-score   support

           0       0.99      0.93      0.96    118621
           1       0.73      0.95      0.83      7959
           2       0.68      0.97      0.80     11438

    accuracy                           0.94    138018
   macro avg       0.80      0.95      0.86    138018
weighted avg       0.95      0.94      0.94    138018



#### 詮釋
雖然recall提升了，但1、2準確率在準確率大幅下降，這表示在1、2類的預測上過於敏感，容易產生誤報，即可能將原本是0的歸類為1或2。未來將先嘗試從調整權重倍數著手，根據文獻指出開根號倒數的調整方法模型表現較佳

In [8]:
# 開根號倒數調整權重
import numpy as np
mod3 = clone(mod_baseline)
mod3 = mod3.fit(
    X_train, y_train,
    XGBclassifier__sample_weight=np.sqrt(compute_sample_weight('balanced', y_train))
)

In [9]:
mod3_train_predict = mod3.predict(X_train)
mod3_test_predict = mod3.predict(X_test)
report_mod3_train = classification_report(y_train, mod3_train_predict)
report_mod3_test = classification_report(y_test, mod3_test_predict)
print(f'train: {report_mod3_train}\n test: {report_mod3_test}')

train:               precision    recall  f1-score   support

           0       0.99      0.96      0.97    473940
           1       0.78      0.94      0.85     31844
           2       0.77      0.94      0.85     46286

    accuracy                           0.95    552070
   macro avg       0.85      0.95      0.89    552070
weighted avg       0.96      0.95      0.96    552070

 test:               precision    recall  f1-score   support

           0       0.99      0.95      0.97    118621
           1       0.78      0.93      0.85      7959
           2       0.77      0.94      0.85     11438

    accuracy                           0.95    138018
   macro avg       0.84      0.94      0.89    138018
weighted avg       0.96      0.95      0.95    138018



In [10]:
# 比較不同模型在測試集的表現
print(f"mod1: {report_test}----\n mod2: {report_mod2_test}----\n mod3:{report_mod3_test}")

mod1:               precision    recall  f1-score   support

           0       0.97      0.99      0.98    118621
           1       0.94      0.82      0.88      7959
           2       0.95      0.80      0.87     11438

    accuracy                           0.97    138018
   macro avg       0.95      0.87      0.91    138018
weighted avg       0.97      0.97      0.97    138018
----
 mod2:               precision    recall  f1-score   support

           0       0.99      0.93      0.96    118621
           1       0.73      0.95      0.83      7959
           2       0.68      0.97      0.80     11438

    accuracy                           0.94    138018
   macro avg       0.80      0.95      0.86    138018
weighted avg       0.95      0.94      0.94    138018
----
 mod3:              precision    recall  f1-score   support

           0       0.99      0.95      0.97    118621
           1       0.78      0.93      0.85      7959
           2       0.77      0.94      0.85     

### 結果解釋
由於模型三在整體而言表現較為穩健 各類別的recall與mod1相比顯著體繩(例如類別1的recall從0.82上升至0.93)，而個別的精確率又比mod3有顯著提升，例如類別2的精確率(precision從0.68到0.77)。整體而言macro f1也比mod表現得叫號(0.89)，故選擇開根號倒數為加權。然實務場景須依造不同需求而定
接著進行學習率與n_estimators的調整，以更高的精準度組合，同時避免過擬合

In [86]:
#　嘗試降低學習率，提升精準度同時計算最佳的n_estimators
learning_rates = [0.1, 0.05, 0.02, 0.01]
# 設定驗證集
X_train_sub, X_train_val, y_train_sub, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)
X_train_sub_proc = preprocessor.fit_transform(X_train_sub)
X_train_val_proc = preprocessor.transform(X_train_val)
X_test_proc = preprocessor.transform(X_test)
from xgboost import XGBClassifier
results = {}
for lr in learning_rates:
    print(f'\n==============learning rate {lr}===================')
    mod4 = XGBClassifier( 
            n_estimators = 3000,
            learning_rate = lr,
            max_depth = 6, 
            subsample = 1.0,
            colsample_bytree = 1.0,
            gamma = 0, 
            reg_alpha = 0,
            reg_lambda = 1,
            early_stopping_rounds = 30,
            eval_metric="mlogloss",
            random_state=42,

            tree_method = 'hist',
            n_jobs = -2
    )
    mod4.fit(
        X_train_sub_proc, y_train_sub,
        sample_weight = np.sqrt(compute_sample_weight('balanced', y_train_sub)),
        eval_set = [(X_train_val_proc, y_val)],
        sample_weight_eval_set=[np.sqrt(compute_sample_weight('balanced', y_val))],
        verbose = 100
    )
    best_tree = mod4.best_iteration
    # test
    mod4_predict = mod4.predict(X_test_proc)
    mod4_report = classification_report(y_test,mod4_predict)
    results[lr] = {
        'learning rate': lr,
        'n_estimators': best_tree,
        'report': mod4_report
    }


==============learning rate 0.1===================
[0]	validation_0-mlogloss:0.78403
[100]	validation_0-mlogloss:0.14770
[200]	validation_0-mlogloss:0.14688
[248]	validation_0-mlogloss:0.14690

==============learning rate 0.05===================
[0]	validation_0-mlogloss:0.84375
[100]	validation_0-mlogloss:0.15187
[200]	validation_0-mlogloss:0.14775
[300]	validation_0-mlogloss:0.14710
[400]	validation_0-mlogloss:0.14684
[489]	validation_0-mlogloss:0.14685

==============learning rate 0.02===================
[0]	validation_0-mlogloss:0.88086
[100]	validation_0-mlogloss:0.21347
[200]	validation_0-mlogloss:0.15706
[300]	validation_0-mlogloss:0.15014
[400]	validation_0-mlogloss:0.14849
[500]	validation_0-mlogloss:0.14779
[600]	validation_0-mlogloss:0.14736
[700]	validation_0-mlogloss:0.14715
[800]	validation_0-mlogloss:0.14703
[900]	validation_0-mlogloss:0.14691
[1000]	validation_0-mlogloss:0.14683
[1081]	validation_0-mlogloss:0.14680

==============learning rate 0.01===================
[

In [ ]:
for lr, info in results.items():
    print("=" * 65),
    print(
        f"🚀 Learning Rate: {lr:<5} | 🌲 Best n_estimators: {info['n_estimators']} 棵樹"
    ),
    print("=" * 65)
    # 印出原本完整的 classification_report 表格
    print(info["report"])
    print("\n")

🚀 Learning Rate: 0.1   | 🌲 Best n_estimators: 218 棵樹
              precision    recall  f1-score   support

           0       0.99      0.96      0.97    118621
           1       0.79      0.93      0.85      7959
           2       0.78      0.93      0.85     11438

    accuracy                           0.95    138018
   macro avg       0.85      0.94      0.89    138018
weighted avg       0.96      0.95      0.96    138018



🚀 Learning Rate: 0.05  | 🌲 Best n_estimators: 459 棵樹
              precision    recall  f1-score   support

           0       0.99      0.96      0.97    118621
           1       0.79      0.93      0.85      7959
           2       0.78      0.93      0.85     11438

    accuracy                           0.96    138018
   macro avg       0.85      0.94      0.89    138018
weighted avg       0.96      0.96      0.96    138018



🚀 Learning Rate: 0.02  | 🌲 Best n_estimators: 1051 棵樹
              precision    recall  f1-score   support

           0       

#### 結果解釋
選擇模型學習力0.05，樹459的模型，在四組學習率測試中，學習率0.05是唯一在測試集上達到96%整體準確率 的組合（其餘學習率均為 95%，展現出最佳的整體泛化能力。觀察Validation mLogloss的收斂過程，當學習率為0.05且樹數量達489棵時，Validation Loss已收斂至0.14685。後續將學習率進一步調低至 0.01（樹數量達 1873 棵），Validation Loss最終仍停留在 0.14685，顯示模型收斂已達局部最佳解。數據顯示，學習率從0.05降至0.01所額外增加的1,380多棵決策樹，並未對 Validation Loss帶來任何實質下降；相反地，過多的決策樹，反而導致測試集上的預測能力微幅下降。




In [11]:
# 調整樹深度，使用網格搜尋
from sklearn.model_selection import GridSearchCV
import numpy as np

from sklearn.utils.class_weight import compute_sample_weight
fina_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('XGB', XGBClassifier(
        n_estimators = 459,
        learning_rate = 0.05,
        max_depth = 6, 
        subsample = 1.0,
        colsample_bytree = 1.0,
        gamma = 0, 
        reg_alpha = 0,
        reg_lambda = 1,
        random_state=42,
        tree_method = 'hist',
        n_jobs=1
    ))    
])

parm_grid = {
    # 控制樹深度
    'XGB__max_depth': [4, 5, 6, 7],
    'XGB__min_child_weight': [1, 3, 5, 10, 20],
}

grid_search = GridSearchCV(
    estimator = fina_model,
    param_grid = parm_grid,
    scoring = 'neg_log_loss',
    cv = 3,
    verbose=3,
    n_jobs=-1   
)
grid_search.fit(X_train, y_train, 
                XGB__sample_weight=np.sqrt(compute_sample_weight('balanced', y_train)))

Fitting 3 folds for each of 20 candidates, totalling 60 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'XGB__max_depth': [4, 5, ...], 'XGB__min_child_weight': [1, 3, ...]}"
,scoring,'neg_log_loss'
,n_jobs,-1
,refit,True
,cv,3
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [14]:
print(grid_search.best_params_)
predict_grid_train = grid_search.predict(X_train)
predict_grid_test = grid_search.predict(X_test)
print(classification_report(y_train, predict_grid_train))
print(classification_report(y_test, predict_grid_test))


{'XGB__max_depth': 7, 'XGB__min_child_weight': 1}
              precision    recall  f1-score   support

           0       0.99      0.96      0.98    473940
           1       0.81      0.95      0.87     31844
           2       0.82      0.95      0.88     46286

    accuracy                           0.96    552070
   macro avg       0.87      0.95      0.91    552070
weighted avg       0.97      0.96      0.96    552070

              precision    recall  f1-score   support

           0       0.99      0.96      0.97    118621
           1       0.79      0.93      0.85      7959
           2       0.79      0.93      0.85     11438

    accuracy                           0.96    138018
   macro avg       0.86      0.94      0.89    138018
weighted avg       0.96      0.96      0.96    138018



In [16]:
# 調整樹深度，使用網格搜尋
from sklearn.model_selection import GridSearchCV
import numpy as np

from sklearn.utils.class_weight import compute_sample_weight
fina_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('XGB', XGBClassifier(
        n_estimators = 459,
        learning_rate = 0.05,
        max_depth = 7,
        min_child_weight = 1, 
        subsample = 1.0,
        colsample_bytree = 1.0,
        gamma = 0, 
        reg_alpha = 0,
        reg_lambda = 1,
        random_state=42,
        tree_method = 'hist',
        n_jobs=1
    ))    
])

parm_grid = {
    # 控制抽樣比例
    'XGB__subsample': [0.6, 0.8, 1.0],
    'XGB__colsample_bytree':[0.6, 0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator = fina_model,
    param_grid = parm_grid,
    scoring = 'neg_log_loss',
    cv = 3,
    verbose=3,
    n_jobs=2   
)
grid_search.fit(X_train, y_train, 
                XGB__sample_weight=np.sqrt(compute_sample_weight('balanced', y_train)))

Fitting 3 folds for each of 9 candidates, totalling 27 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'XGB__colsample_bytree': [0.6, 0.8, ...], 'XGB__subsample': [0.6, 0.8, ...]}"
,scoring,'neg_log_loss'
,n_jobs,2
,refit,True
,cv,3
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [17]:
print(grid_search.best_params_)
predict_grid_train = grid_search.predict(X_train)
predict_grid_test = grid_search.predict(X_test)
print(classification_report(y_train, predict_grid_train))
print(classification_report(y_test, predict_grid_test))

{'XGB__colsample_bytree': 1.0, 'XGB__subsample': 0.8}
              precision    recall  f1-score   support

           0       0.99      0.97      0.98    473940
           1       0.82      0.95      0.88     31844
           2       0.82      0.95      0.88     46286

    accuracy                           0.96    552070
   macro avg       0.88      0.95      0.91    552070
weighted avg       0.97      0.96      0.96    552070

              precision    recall  f1-score   support

           0       0.99      0.96      0.97    118621
           1       0.80      0.93      0.86      7959
           2       0.79      0.93      0.85     11438

    accuracy                           0.96    138018
   macro avg       0.86      0.94      0.89    138018
weighted avg       0.96      0.96      0.96    138018



In [6]:
from sklearn.model_selection import GridSearchCV
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

fina_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('XGB', XGBClassifier(
        n_estimators = 459,
        learning_rate = 0.05,
        max_depth = 7,
        min_child_weight = 1, 
        subsample = 0.8,
        colsample_bytree = 1.0,
        gamma = 0, 
        reg_alpha = 0,
        reg_lambda = 1,
        random_state=42,
        tree_method = 'hist',
        n_jobs=1
    ))    
])

fina_model.fit(X, y, XGB__sample_weight=np.sqrt(compute_sample_weight('balanced', y)))

,steps,"[('preprocessor', ...), ('XGB', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [7]:
import joblib
joblib.dump(fina_model, 'health_xgb_mod_baseline.pkl')

['health_xgb_mod_baseline.pkl']

In [ ]:
test_df = pd.read_csv('test.csv')
id = test_df.loc[:, 'id']
#　drfine predict data
X_test_df = test_df.loc[:, [c for c in test_df.columns if c not in ['id']]]
predict = fina_model.predict(X_test_df)


In [ ]:
import pandas as pd
from xgboost import XGBClassifier

final_model = XGBClassifier(
    n_estimators=459,  
    learning_rate=0.05,  
    max_depth=6,
    subsample=1.0,
    colsample_bytree=1.0,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    eval_metric="mlogloss",
    n_jobs=-2,
    random_state=42,
    tree_method="hist",
)

final_model.fit(X_train_sub_proc, y_train_sub, sample_weight=np.sqrt(compute_sample_weight('balanced', y_train_sub)))


importance = final_model.get_booster().get_score(importance_type="gain")

try:
    feature_names = preprocessor.get_feature_names_out()
    mapped_importance = {
        feature_names[int(k.replace("f", ""))]: v for k, v in importance.items()
    }
except Exception:
    mapped_importance = importance

df_imp = (
    pd.DataFrame(
        list(mapped_importance.items()), columns=["Feature", "Importance_Gain"]
    )
    .sort_values("Importance_Gain", ascending=False)
    .reset_index(drop=True)
)

display(df_imp.head(20))
# 取得各特徵對模型的Gain
importance = final_model.get_booster().get_score(importance_type="gain")
# 整理成表格並排序
import pandas as pd

df_imp = pd.DataFrame(
    list(importance.items()), columns=["Feature", "Gain"]
).sort_values("Gain", ascending=False)
display(df_imp)

=== 📊 特徵重要性（Feature Importance - Gain）前 20 名 ===


,Feature,Importance_Gain
0,cat__physical_activity_level_active,1357.247559
1,cat__stress_level_low,1132.951172
2,cat__stress_level_medium,960.191467
3,cat__stress_level_high,591.608459
4,cat__stress_level_nan,351.975952
5,cat__physical_activity_level_nan,247.294403
6,num__sleep_duration,208.665512
7,cat__sleep_quality_good,24.282127
8,cat__sleep_quality_poor,16.031153
9,num__bmi,12.892204


,Feature,Gain
19,cat__physical_activity_level_active,1357.247559
12,cat__stress_level_low,1132.951172
13,cat__stress_level_medium,960.191467
11,cat__stress_level_high,591.608459
14,cat__stress_level_nan,351.975952
22,cat__physical_activity_level_nan,247.294403
0,num__sleep_duration,208.665512
16,cat__sleep_quality_good,24.282127
17,cat__sleep_quality_poor,16.031153
2,num__bmi,12.892204
